In [3]:
import sys, os; sys.path.append(os.path.dirname(os.getcwd())) 
import numpy as np
import numpy as np
import os
np.random.seed(1000)
# Function to simulate multivariate regression data
def simulate_data(q, sigma, n_predictors, sample_size, beta):
    X_designs = []
    Ys = []
    
    for _ in range(sample_size):
        # Generate the design matrix X, adding the column of ones for the intercept term
        X_design = np.random.uniform(0, 1, size=(1, n_predictors))
        
        # Generate the error term epsilon, which follows a multivariate normal distribution
        epsilon = np.random.multivariate_normal(np.zeros(q), sigma * np.eye(q))
        
        # Calculate the response vector Y = Beta * X + epsilon
        Y = X_design @ beta + epsilon
        
        X_designs.append(X_design)
        Ys.append(Y)
    
    return np.array(X_designs).reshape(sample_size,n_predictors), np.array(Ys).reshape(sample_size,q)

# Function to simulate the data for multiple scenarios (sigma, sample_size)
def simulate_for_scenarios(q, sigma_values, n_predictors, sample_sizes, beta):
    results = {}
    
    for sigma in sigma_values:
        for sample_size in sample_sizes:
            X_design, Y = simulate_data(q, sigma, n_predictors, sample_size, beta)
            results[(sigma, sample_size)] = {'X': X_design, 'Y': Y}
    
    return results


# Set parameters for the regression scenario
q = 15 # Dimensionality of the data (q features)
n_predictors = 10  # Number of predictors
sigma_values = [0.1, 0.2, 0.3, 0.4, 0.5]  # Different sigma values
sample_sizes = [50, 100, 200, 500]  # Different sample sizes

# Sample beta from a multivariate normal distribution
beta = np.random.multivariate_normal(np.ones(n_predictors * q), 2*np.eye(n_predictors * q)).reshape(n_predictors, q) # n_predictors*q
# Simulate the regression data for all scenarios
results = simulate_for_scenarios(q, sigma_values, n_predictors, sample_sizes, beta)


In [4]:
results = {}

for sigma in sigma_values:
    for sample_size in sample_sizes:
        X_design, Y = simulate_data(q, sigma, n_predictors, sample_size, beta)
        results[(sigma, sample_size)] = {'X': X_design, 'Y': Y}



In [5]:
X_designs = []
Ys = []

for _ in range(sample_size):
    # Generate the design matrix X, adding the column of ones for the intercept term
    X_design = np.random.uniform(0, 1, size=(1, n_predictors))
    
    # Generate the error term epsilon, which follows a multivariate normal distribution
    epsilon = np.random.multivariate_normal(np.zeros(q), sigma * np.eye(q))
    
    # Calculate the response vector Y = Beta * X + epsilon
    Y = X_design @ beta + epsilon
    
    X_designs.append(X_design)
    Ys.append(Y)
np.array(X_designs).reshape(sample_size,n_predictors), np.array(Ys).reshape(sample_size,q)

(array([[0.74471399, 0.21257449, 0.32930505, ..., 0.59289969, 0.91010811,
         0.11118491],
        [0.4646033 , 0.60118531, 0.83139757, ..., 0.33695185, 0.03718822,
         0.92793654],
        [0.78614984, 0.64841596, 0.19475617, ..., 0.24415679, 0.17554621,
         0.19762206],
        ...,
        [0.50787647, 0.45112593, 0.50981407, ..., 0.34317337, 0.96441323,
         0.28370911],
        [0.09153402, 0.26665149, 0.89338642, ..., 0.07548011, 0.82027117,
         0.83886212],
        [0.0954626 , 0.85740178, 0.17425526, ..., 0.01129886, 0.39557401,
         0.84310052]]),
 array([[ 0.26754794,  3.75387313,  0.5439763 , ...,  5.68064843,
          2.08005348,  1.42702324],
        [ 3.73539765,  8.69816162,  5.64482217, ...,  8.26139305,
          5.95917902,  7.96572755],
        [ 0.95056288,  2.89252017,  3.8261928 , ...,  2.67885934,
          3.42520035,  6.29618825],
        ...,
        [ 2.52184191,  4.4130218 ,  3.48937058, ...,  8.96453172,
          4.3857697 ,  4

In [6]:
X_designs = []
Ys = []

for _ in range(sample_size):
    # Generate the design matrix X, adding the column of ones for the intercept term
    X_design = np.random.uniform(0, 1, size=(1, n_predictors))
    
    # Generate the error term epsilon, which follows a multivariate normal distribution
    epsilon = np.random.multivariate_normal(np.zeros(q), sigma * np.eye(q))
    
    # Calculate the response vector Y = Beta * X + epsilon
    Y = X_design @ beta + epsilon
    
    X_designs.append(X_design)
    Ys.append(Y)
    
# Convert lists to arrays for easier manipulation
X_designs = np.array(X_designs).reshape(sample_size, n_predictors)
Ys = np.array(Ys).reshape(sample_size, q)

# Center the predictors
X_designs -= X_designs.mean(axis=0)

# Center the responses
Ys -= Ys.mean(axis=0)

In [7]:
from pyfrechet.metric_spaces import Euclidean, MetricData
# Convert response to MetricData
M = Euclidean(dim=15)
y = MetricData(M, Ys)

INFO: Using numpy backend


In [8]:
import numpy as np
import pickle
# from sklearn.model_selection import cross_val_score
# from sklearn.model_selection import GridSearchCV
# from sklearn import neighbors, clone


from pyfrechet.metric_spaces import MetricData, Euclidean
# from pyfrechet.regression.frechet_regression import LocalFrechet, GlobalFrechet
# from pyfrechet.regression.kernels import NadarayaWatson, gaussian, epanechnikov
# from pyfrechet.regression.knn import KNearestNeighbours
from pyfrechet.regression.bagged_regressor import BaggedRegressor
from pyfrechet.regression.trees import Tree
from pyfrechet.metrics import mse

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.utils import _array_indexing

import os
import sys
import pickle
import numpy as np
from joblib import Parallel, delayed
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer


def tune_forest(X, y):
    """ Perform hyperparameter tuning using GridSearchCV. """
    base = Tree(split_type='2means', impurity_method='cart')
    forest = BaggedRegressor(estimator=base, n_estimators=20, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)
    
    tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=5, n_jobs=-1, verbose=4)
    tuned_forest.fit(X, y)
    return tuned_forest.best_estimator_

In [9]:
# Define parameter grid for tuning
param_grid = {
    'estimator__min_split_size': [1, 3, 5, 7, 10, 15],}

# Custom scorer (negative mean squared error, assuming mse is defined elsewhere)
neg_mse = make_scorer(mse, greater_is_better=False)

tune_forest(X_designs, Ys)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend
INFO: Using numpy backend


[CV 2/5] END .........estimator__min_split_size=1;, score=nan total time=   0.3s
[CV 1/5] END .........estimator__min_split_size=1;, score=nan total time=   0.3s
[CV 3/5] END .........estimator__min_split_size=1;, score=nan total time=   0.2s
[CV 2/5] END .........estimator__min_split_size=3;, score=nan total time=   0.3s
[CV 4/5] END .........estimator__min_split_size=1;, score=nan total time=   0.4s
[CV 1/5] END .........estimator__min_split_size=3;, score=nan total time=   0.4s
[CV 5/5] END .........estimator__min_split_size=1;, score=nan total time=   0.4s
[CV 4/5] END .........estimator__min_split_size=3;, score=nan total time=   0.0s
[CV 5/5] END .........estimator__min_split_size=3;, score=nan total time=   0.0s
[CV 3/5] END .........estimator__min_split_size=3;, score=nan total time=   0.4s
[CV 2/5] END .........estimator__min_split_size=5;, score=nan total time=   0.0s
[CV 1/5] END .........estimator__min_split_size=5;, score=nan total time=   0.0s
[CV 3/5] END .........estima

ValueError: 
All the 30 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/Diego/miniconda3/envs/pballs/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/Diego/Desktop/Codigo/repo_edu_pyfrechet/pyfrechet/pyfrechet/regression/bagged_regressor.py", line 85, in fit
    return self._fit_seq(X, y) if self.n_jobs == 1 or not self.n_jobs else self._fit_par(X, y)
  File "/Users/Diego/Desktop/Codigo/repo_edu_pyfrechet/pyfrechet/pyfrechet/regression/bagged_regressor.py", line 73, in _fit_seq
    self.estimators = [ self._fit_est(X, y) for _ in range(self.n_estimators)]
  File "/Users/Diego/Desktop/

In [10]:
y = MetricData(M, Ys)

In [11]:
base = Tree(split_type='2means', impurity_method='cart')
forest = BaggedRegressor(estimator=base, n_estimators=1, bootstrap_fraction=1, bootstrap_replace=True, n_jobs=1)

tuned_forest = GridSearchCV(estimator=forest, param_grid=param_grid, scoring=neg_mse, cv=2, n_jobs=-1, verbose=1)
tuned_forest.fit(X_designs, y)
tuned_forest.best_estimator_

Fitting 2 folds for each of 6 candidates, totalling 12 fits


BaggedRegressor(bootstrap_fraction=1, bootstrap_replace=True,
                estimator=Tree(min_split_size=10, split_type='2means'),
                n_estimators=1, n_jobs=1)

In [12]:
from sklearn.model_selection import train_test_split
M = Euclidean(dim=15)
y = np.ones((10,15))
X = np.ones((10,12))
y = MetricData(M, y)
train_test_split(X, y, test_size=0.2)

[array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]),
 array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]),